# Unit IV — Clinical Decision Support & Real-World Data
**Pharmakaksha · Learn · Grow · Achieve** — BP703T AI in Clinical Applications (B.Pharm Sem VII, PCI NEP 2020)

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ajreddys/learn-pharmakaksha-site/blob/main/content/BP703T%20AI%20in%20Clinical%20Applications%20%28Theory%29/BP703T_Unit4_Clinical_Decision_Support.ipynb)

Every example from the Unit IV notes, slides and video, in order, then lab exercises with solutions. Data: `ehr_patients.csv`, `ehr_encounters.csv`, `ehr_labs.csv`, `ehr_meds.csv` (fictional teaching data).

## Using learn.pharmakaksha.com
- **First run is slow.** Python starts inside your browser; the first cell can take about 10 seconds. Later cells are fast.
- **Kernel indicator.** The circle at the top right is filled while Python is busy. Wait for it to clear.
- **Stuck?** Use *Kernel → Restart Kernel*, then run the cells again from the top.
- **Saving.** Your changes are saved in this browser only. Use *File → Download* to keep a copy.
- **Start over.** Delete your copy from the file list on the left and reopen the notebook from the course folder.

In [ ]:
import os
import numpy as np
import pandas as pd
BASE = "https://raw.githubusercontent.com/ajreddys/learn-pharmakaksha-site/main/content/BP703T%20AI%20in%20Clinical%20Applications%20%28Theory%29/"

def load(name):
    """Read a practice CSV: local copy first, then the course's GitHub folder."""
    if os.path.exists(name):
        return pd.read_csv(name)
    try:
        return pd.read_csv(BASE + name)
    except Exception:
        from pyodide.http import open_url   # inside JupyterLite
        return pd.read_csv(open_url(BASE + name))

## 4.1 EHR structure: four linked tables
A fictional hospital extract: `patients` → `encounters` (one per admission) → `labs` and `meds` (many per encounter).

In [ ]:
pat = load("ehr_patients.csv")
enc = load("ehr_encounters.csv")
labs = load("ehr_labs.csv")
meds = load("ehr_meds.csv")
for name, t in [("patients", pat), ("encounters", enc), ("labs", labs), ("meds", meds)]:
    print(f"{name:10s} {t.shape}")
enc.head()

In [ ]:
labs.head()

## 4.5 Real-world data analytics: from tables to one row per encounter
Join, compute, pivot and count.

In [ ]:
enc["admit_date"] = pd.to_datetime(enc["admit_date"])
enc["discharge_date"] = pd.to_datetime(enc["discharge_date"])
df = enc.merge(pat, on="patient_id")
df["age"] = df["admit_date"].dt.year - df["birth_year"]
df["los_days"] = (df["discharge_date"] - df["admit_date"]).dt.days
df = df.sort_values(["patient_id", "admit_date"])
df["prior_admissions"] = df.groupby("patient_id").cumcount()

lab_wide = labs.pivot_table(index="encounter_id", columns="test", values="value")
df = df.merge(lab_wide, on="encounter_id", how="left")
df = df.merge(meds.groupby("encounter_id").size().rename("n_meds"), on="encounter_id", how="left")
df[["encounter_id", "age", "diagnosis", "los_days", "prior_admissions", "hemoglobin_g_dL", "sodium_mmol_L", "n_meds", "readmit_30d"]].head()

In [ ]:
df.groupby("diagnosis").agg(encounters=("los_days", "size"), mean_los=("los_days", "mean"),
                            readmission_rate=("readmit_30d", "mean")).round(2)

### Missing data: a fact of real-world records

In [ ]:
lab_cols = ["hemoglobin_g_dL", "sodium_mmol_L", "creatinine_mg_dL"]
print(df[lab_cols].isna().mean().round(3))
for c in lab_cols:
    df[c + "_missing"] = df[c].isna().astype(int)
    df[c] = df[c].fillna(df[c].median())

## 4.3 Regression for outcome prediction: length of stay

In [ ]:
from sklearn.linear_model import LinearRegression, LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error, r2_score, roc_auc_score

dx = pd.get_dummies(df["diagnosis"], prefix="dx", dtype=int).drop(columns="dx_Pneumonia")
X = pd.concat([df[["age", "hemoglobin_g_dL", "sodium_mmol_L", "creatinine_mg_dL", "n_meds", "prior_admissions"]], dx], axis=1)
X_tr, X_te, y_tr, y_te = train_test_split(X, df["los_days"], test_size=0.25, random_state=42)
los_model = LinearRegression().fit(X_tr, y_tr)
pred = los_model.predict(X_te)
print("Mean LOS:", round(df["los_days"].mean(), 2), "days")
print("RMSE:", round(mean_squared_error(y_te, pred) ** 0.5, 2), "days | R²:", round(r2_score(y_te, pred), 3))
print("Baseline RMSE (predict the mean):", round(mean_squared_error(y_te, np.full(len(y_te), y_tr.mean())) ** 0.5, 2))
pd.Series(los_model.coef_, index=X.columns).round(2).rename("days per unit")

## 4.4 Classification for risk scoring: 30-day readmission

In [ ]:
Xr = X.copy()
Xr["los_5plus"] = (df["los_days"] >= 5).astype(int)
y = df["readmit_30d"]
Xr_tr, Xr_te, yr_tr, yr_te = train_test_split(Xr, y, test_size=0.3, random_state=42, stratify=y)
readm = LogisticRegression(C=1e6, max_iter=10000).fit(Xr_tr, yr_tr)
p = readm.predict_proba(Xr_te)[:, 1]
rf = RandomForestClassifier(n_estimators=300, min_samples_leaf=10, random_state=42).fit(Xr_tr, yr_tr)
print("Readmission rate:", round(y.mean(), 3))
print("AUC logistic:     ", round(roc_auc_score(yr_te, p), 3))
print("AUC random forest:", round(roc_auc_score(yr_te, rf.predict_proba(Xr_te)[:, 1]), 3))
pd.Series(np.exp(readm.coef_[0]), index=Xr.columns).round(2).rename("odds ratio")

### A bedside points score (0–8)
One point each for: Hb < 12, sodium < 135, length of stay ≥ 5 days, ≥ 8 medicines, heart failure or COPD, age ≥ 75; plus up to 2 points for prior admissions.

In [ ]:
def readmission_score(r):
    s = 0
    s += r["hemoglobin_g_dL"] < 12
    s += r["sodium_mmol_L"] < 135
    s += min(int(r["prior_admissions"]), 2)
    s += r["los_5plus"] == 1
    s += r["n_meds"] >= 8
    s += (r["dx_Heart failure"] == 1) or (r["dx_COPD exacerbation"] == 1)
    s += r["age"] >= 75
    return int(s)

test = Xr_te.copy()
test["score"] = test.apply(readmission_score, axis=1)
test["readmitted"] = yr_te
print("AUC points score:", round(roc_auc_score(yr_te, test["score"]), 3))
test.groupby("score").agg(n=("readmitted", "size"), readmission_rate=("readmitted", "mean")).round(3)

## 4.2 CDSS: how many alerts can a pharmacy team handle?

In [ ]:
rows = []
for t in [3, 4, 5]:
    flag = test["score"] >= t
    rows.append({"alert if score >=": t, "alerts per 100 discharges": round(100 * flag.mean(), 1),
                 "readmissions caught": round(test.loc[flag, "readmitted"].sum() / test["readmitted"].sum(), 2),
                 "PPV": round(test.loc[flag, "readmitted"].mean(), 2)})
pd.DataFrame(rows)

## Worked example: two discharges today

In [ ]:
today = pd.DataFrame([
    {"age": 81, "hemoglobin_g_dL": 10.2, "sodium_mmol_L": 131, "creatinine_mg_dL": 1.6, "n_meds": 11, "prior_admissions": 2,
     "dx_COPD exacerbation": 0, "dx_Heart failure": 1, "dx_Type 2 diabetes": 0, "dx_Urinary tract infection": 0, "los_5plus": 1},
    {"age": 55, "hemoglobin_g_dL": 12.8, "sodium_mmol_L": 138, "creatinine_mg_dL": 0.9, "n_meds": 4, "prior_admissions": 0,
     "dx_COPD exacerbation": 0, "dx_Heart failure": 0, "dx_Type 2 diabetes": 0, "dx_Urinary tract infection": 1, "los_5plus": 0}],
    index=["Mr. D (heart failure)", "Ms. E (UTI)"])[Xr.columns]
today["score"] = today.apply(readmission_score, axis=1)
today["P(readmit)"] = readm.predict_proba(today[Xr.columns])[:, 1].round(3)
today[["score", "P(readmit)"]]

---
# Lab exercises
1. Which diagnosis has the highest readmission rate and the longest mean stay?
2. Fit the LOS model without the lab values. How much do RMSE and R² change?
3. Add `discharge_to` (one-hot) to the readmission model. Does AUC improve?
4. If the pharmacy team can review 20 patients per 100 discharges, which score threshold would you choose, and what share of readmissions would it catch?

## Solutions

In [ ]:
# Lab 1
df.groupby("diagnosis")[["readmit_30d", "los_days"]].mean().round(2).sort_values("readmit_30d", ascending=False)

In [ ]:
# Lab 2
X_nolab = X.drop(columns=lab_cols)
a, b, ya, yb = train_test_split(X_nolab, df["los_days"], test_size=0.25, random_state=42)
m2 = LinearRegression().fit(a, ya)
print("Without labs: RMSE", round(mean_squared_error(yb, m2.predict(b)) ** 0.5, 2), "R²", round(r2_score(yb, m2.predict(b)), 3))

In [ ]:
# Lab 3
Xd = pd.concat([Xr, pd.get_dummies(df["discharge_to"], prefix="to", dtype=int).drop(columns="to_Home")], axis=1)
a, b, ya, yb = train_test_split(Xd, y, test_size=0.3, random_state=42, stratify=y)
m3 = LogisticRegression(C=1e6, max_iter=10000).fit(a, ya)
print("AUC with discharge destination:", round(roc_auc_score(yb, m3.predict_proba(b)[:, 1]), 3))

In [ ]:
# Lab 4
for t in range(0, 9):
    flag = test["score"] >= t
    if 100 * flag.mean() <= 20:
        print("threshold", t, "| alerts per 100:", round(100 * flag.mean(), 1),
              "| caught:", round(test.loc[flag, "readmitted"].sum() / test["readmitted"].sum(), 2)); break

**Teaching data only.** The hospital, patients and records are fictional.